In [ ]:
import numpy as np
import pandas as pd

def preprocess_data(df):
    """Tính doanh thu thuần và chuẩn hóa cột thời gian"""
    df['OrderDate'] = pd.to_datetime(df['OrderDate'])
    df['YearMonth'] = df['OrderDate'].dt.to_period('M')

    # Net Revenue sau chiết khấu
    if 'Discount' in df.columns:
        df['NetRevenue'] = (
            df['UnitPrice'] * df['Quantity'] * (1 - df['Discount'])
        )
    else:
        df['NetRevenue'] = df['UnitPrice'] * df['Quantity']
    return df


# ==============================================================================
# 1. HÀM TÍNH TOÁN KPI THEO MỌI CẤP PHÂN CẤP (Region, City, District, OfficeID)
# ==============================================================================
def calculate_hierarchy_kpis(df, group_col):
    """Tính các KPI: Net Revenue, Quantity, Đơn hàng, Khách hàng, AOV, Rev/Cust, Tỷ trọng"""
    total_system_revenue = df['NetRevenue'].sum()

    agg_df = (
        df.groupby(group_col)
        .agg(
            NetRevenue=('NetRevenue', 'sum'),
            TotalQuantity=('Quantity', 'sum'),
            InvoiceCount=('OrderID', 'nunique'),
            CustomerCount=('CustomerID', 'nunique'),
        )
        .reset_index()
    )

    # Các chỉ số bình quân và tỷ trọng đóng góp nội bộ
    agg_df['AOV'] = agg_df['NetRevenue'] / agg_df['InvoiceCount']
    agg_df['Rev_Per_Customer'] = (
        agg_df['NetRevenue'] / agg_df['CustomerCount']
    )
    agg_df['Revenue_Contribution_Pct'] = (
        agg_df['NetRevenue'] / total_system_revenue
    ) * 100

    # Sắp xếp xếp hạng doanh thu giảm dần
    agg_df = agg_df.sort_values(by='NetRevenue', ascending=False).reset_index(
        drop=True
    )
    return agg_df


# ==============================================================================
# 2. TÍNH MOM GROWTH & PHÂN LOẠI MA TRẬN REVENUE - GROWTH
# ==============================================================================
def calculate_growth_and_matrix(df, group_col):
    """Tính tốc độ tăng trưởng MoM gần nhất và phân loại ma trận 4 góc phần tư"""
    # Gom nhóm theo đơn vị phân cấp và từng tháng
    monthly_df = (
        df.groupby([group_col, 'YearMonth'])['NetRevenue'].sum().reset_index()
    )
    monthly_df['MoM_Growth'] = (
        monthly_df.groupby(group_col)['NetRevenue'].pct_change() * 100
    )

    # Lấy MoM Growth của tháng gần nhất
    latest_month = monthly_df['YearMonth'].max()
    latest_growth = monthly_df[monthly_df['YearMonth'] == latest_month][
        [group_col, 'MoM_Growth']
    ]

    # Kết hợp với tổng doanh thu
    kpi_df = calculate_hierarchy_kpis(df, group_col)
    merged_df = pd.merge(kpi_df, latest_growth, on=group_col, how='left')
    merged_df['MoM_Growth'] = merged_df['MoM_Growth'].fillna(0)

    # Xác định ngưỡng phân vị (Median) để chia ma trận
    rev_median = merged_df['NetRevenue'].median()
    growth_median = merged_df['MoM_Growth'].median()

    def classify_matrix(row):
        high_rev = row['NetRevenue'] >= rev_median
        high_growth = row['MoM_Growth'] >= growth_median

        if high_rev and high_growth:
            return 'Thị trường trọng điểm'
        elif high_rev and not high_growth:
            return 'Cần bảo vệ và tái kích hoạt'
        elif not high_rev and high_growth:
            return 'Thị trường tiềm năng'
        else:
            return 'Xem xét tối ưu hoặc thu hẹp'

    merged_df['Strategic_Segment'] = merged_df.apply(classify_matrix, axis=1)
    return merged_df


# ==============================================================================
# 3. ĐÁNH GIÁ HIỆU QUẢ VĂN PHÒNG / ĐẠI LÝ (Repeat Rate, Scale Comparison)
# ==============================================================================
def evaluate_offices_agencies(df):
    """Đánh giá chi tiết cấp Văn phòng/Đại lý: Repeat Rate, Rev/Cust, AOV"""
    base_kpi = calculate_hierarchy_kpis(
        df, ['Region', 'City', 'District', 'OfficeID']
    )

    # Tính Repeat Rate cho từng Office
    cust_orders = (
        df.groupby(['OfficeID', 'CustomerID'])['OrderID']
        .nunique()
        .reset_index()
    )
    repeat_stats = (
        cust_orders.groupby('OfficeID')
        .agg(
            TotalCust=('CustomerID', 'count'),
            RepeatCust=('OrderID', lambda x: (x > 1).sum()),
        )
        .reset_index()
    )
    repeat_stats['Repeat_Rate_Pct'] = (
        repeat_stats['RepeatCust'] / repeat_stats['TotalCust']
    ) * 100

    # Ghép bảng đánh giá
    office_eval = pd.merge(
        base_kpi,
        repeat_stats[['OfficeID', 'Repeat_Rate_Pct']],
        on='OfficeID',
        how='left',
    )

    # Phân nhóm quy mô theo số lượng khách hàng để so sánh tương đồng
    office_eval['Scale_Group'] = pd.qcut(
        office_eval['CustomerCount'],
        q=3,
        labels=['Quy mô nhỏ', 'Quy mô vừa', 'Quy mô lớn'],
    )

    # Xếp hạng nội bộ trong cùng Region/City theo hiệu quả Rev_Per_Customer
    office_eval['Rank_In_City'] = office_eval.groupby('City')[
        'Rev_Per_Customer'
    ].rank(ascending=False, method='dense')

    return office_eval.sort_values(by=['City', 'Rank_In_City'])


# ==============================================================================
# THỰC THI (MAIN)
# ==============================================================================
if __name__ == '__main__':
    # Tạo dữ liệu mẫu kiểm thử
    np.random.seed(42)
    sample_data = {
        'OrderID': np.random.randint(1000, 1050, size=200),
        'OrderDate': pd.date_range(start='2026-01-01', periods=200, freq='D'),
        'CustomerID': np.random.randint(1, 40, size=200),
        'Region': np.random.choice(['Miền Nam', 'Miền Bắc'], size=200),
        'City': np.random.choice(['TP.HCM', 'Hà Nội', 'Đà Nẵng'], size=200),
        'District': np.random.choice(
            ['Quận 1', 'Quận 3', 'Cầu Giấy', 'Ba Đình'], size=200
        ),
        'OfficeID': np.random.choice(
            ['Agency_A', 'Agency_B', 'Agency_C', 'Agency_D'], size=200
        ),
        'UnitPrice': np.random.uniform(50, 500, size=200),
        'Quantity': np.random.randint(1, 10, size=200),
        'Discount': np.random.choice([0, 0.05, 0.1], size=200),
    }
    df = pd.DataFrame(sample_data)
    df = preprocess_data(df)

    # 1. Báo cáo phân cấp City
    city_matrix = calculate_growth_and_matrix(df, 'City')
    print('--- MA TRẬN CHIẾN LƯỢC THEO CITY ---')
    print(
        city_matrix[
            [
                'City',
                'NetRevenue',
                'MoM_Growth',
                'Revenue_Contribution_Pct',
                'Strategic_Segment',
            ]
        ]
    )

    # 2. Đánh giá hiệu quả văn phòng/đại lý
    office_report = evaluate_offices_agencies(df)
    print('\n--- HIỆU QUẢ VĂN PHÒNG / ĐẠI LÝ ---')
    print(
        office_report[
            [
                'OfficeID',
                'City',
                'Scale_Group',
                'Rev_Per_Customer',
                'Repeat_Rate_Pct',
                'Rank_In_City',
            ]
        ].head()
    )